# dqf — Databricks Notebook Example

This notebook demonstrates how to use `dqf` inside a **Databricks notebook** with the `DatabricksNotebookAdapter`.

The adapter requires no configuration — it discovers the `spark` session that Databricks pre-injects into every notebook automatically.

## Scenario

We validate a customer features table stored in a Delta Lake catalog:

- **Universe** — `main.lending.customers` — defines the population of active customers
- **Variables** — `main.lending.customer_features` — monthly feature snapshot joined to the universe
- **Target** — `main.lending.loan_outcomes` — binary default label, monitored for drift


## 1. Install dqf

Run once per cluster. Skip if already installed on the cluster init script.

In [ ]:
%pip install dqf

## 2. Imports

In [ ]:
import dqf
from dqf.checks.cross_sectional.range_check import RangeCheck
from dqf.checks.pipeline import CheckPipeline

## 3. Create the adapter

`DatabricksNotebookAdapter` takes no arguments. It discovers the `spark` session injected by the Databricks runtime on the first query.

In [ ]:
adapter = dqf.DatabricksNotebookAdapter()

## 4. Define the universe

The universe is the **source of truth for the population**. All quality metrics are measured against it, not against the raw feature table row count.

Here we scope to active customers with at least one loan in the last 12 months.

In [ ]:
UNIVERSE_SQL = """
SELECT customer_id
FROM main.lending.customers
WHERE status = 'active'
  AND last_loan_date >= add_months(current_date(), -12)
"""

universe = dqf.UniverseDataset(
    sql=UNIVERSE_SQL,
    primary_key=["customer_id"],
    adapter=adapter,
)

print(f"Universe size: {universe.size()}")

## 5. Define the variables dataset

The variables dataset is **left-joined to the universe**, so customers with no feature row appear as nulls rather than being silently excluded from quality metrics.

In [ ]:
FEATURES_SQL = """
SELECT
    customer_id,
    snapshot_date,
    credit_score,
    debt_to_income,
    num_open_accounts,
    income_band,
    has_prior_default
FROM main.lending.customer_features
WHERE snapshot_date = current_date()
"""

dataset = dqf.VariablesDataset(
    sql=FEATURES_SQL,
    primary_key=["customer_id"],
    universe=universe,
    join_keys={"customer_id": "customer_id"},
    adapter=adapter,
    variables=[
        dqf.Variable(
            name="customer_id",
            dtype=dqf.DataType.IDENTIFIER,
            role=dqf.VariableRole.IDENTIFIER,
        ),
        dqf.Variable(
            name="credit_score",
            dtype=dqf.DataType.NUMERIC_CONTINUOUS,
        ),
        dqf.Variable(
            name="debt_to_income",
            dtype=dqf.DataType.NUMERIC_CONTINUOUS,
        ),
        dqf.Variable(
            name="num_open_accounts",
            dtype=dqf.DataType.NUMERIC_DISCRETE,
        ),
        dqf.Variable(
            name="income_band",
            dtype=dqf.DataType.CATEGORICAL,
        ),
        dqf.Variable(
            name="has_prior_default",
            dtype=dqf.DataType.BOOLEAN,
        ),
    ],
)

## 6. Build the resolver

We use `build_default_resolver` with a time dimension so that longitudinal checks (trend, structural break) are included for numeric features.

We also register a domain-specific rule: `credit_score` must stay within the FICO bounds [300, 850].

In [ ]:
resolver = dqf.build_default_resolver(
    time_field="snapshot_date",
    period="month",
    null_threshold=0.02,  # fail features with > 2% nulls
    max_categorical_cardinality=10,  # income_band should have few distinct values
)

# Domain rule: credit_score must always be a valid FICO score
resolver.register(
    predicate=lambda v: v.name == "credit_score",
    pipeline_factory=lambda: CheckPipeline(
        [("range", RangeCheck(min_value=300, max_value=850, severity=dqf.Severity.FAILURE))]
    ),
    priority=50,  # beats the default NUMERIC_CONTINUOUS rule at priority 15
)

## 7. Run validation

In [ ]:
report = dataset.run_validation(resolver, dataset_name="customer_features")

print(f"Overall status : {report.overall_status.value}")
print(f"Universe size  : {report.universe_size}")

## 8. Inspect results

In [ ]:
df = report.to_dataframe()
display(df)

In [ ]:
if report.failed_variables():
    print("FAILED variables:", report.failed_variables())
else:
    print("All variables passed.")

if report.warned_variables():
    print("Variables with warnings:", report.warned_variables())

## 9. HTML report with embedded plots

`render()` produces a self-contained HTML report with one section per variable, including distribution plots where available.

In [ ]:
from IPython.display import HTML

HTML(report.render())

## 10. Target variable drift monitoring (optional)

To monitor the loan default rate for concept drift, declare `has_prior_default` (or a proper target column) with `role=TARGET`. The default resolver then applies `ProportionDriftCheck` automatically.

Here we show the pattern with a dedicated target table.

In [ ]:
OUTCOMES_SQL = """
SELECT
    customer_id,
    outcome_date,
    is_default
FROM main.lending.loan_outcomes
WHERE outcome_date >= add_months(current_date(), -12)
"""

target_dataset = dqf.VariablesDataset(
    sql=OUTCOMES_SQL,
    primary_key=["customer_id"],
    universe=universe,
    join_keys={"customer_id": "customer_id"},
    adapter=adapter,
    variables=[
        dqf.Variable(
            name="is_default",
            dtype=dqf.DataType.BOOLEAN,
            role=dqf.VariableRole.TARGET,  # triggers ProportionDriftCheck
        ),
    ],
)

target_resolver = dqf.build_default_resolver(
    time_field="outcome_date",
    period="month",
)

target_report = target_dataset.run_validation(target_resolver, dataset_name="loan_outcomes")

print(f"Target drift status: {target_report.overall_status.value}")

for result in target_report.variable_results.get("is_default", []):
    status = "PASS" if result.passed else "FAIL"
    print(f"  [{status}] {result.check_name:<20} observed={result.observed_value}")
    if result.check_name == "proportion_drift" and result.metadata:
        print(f"         min p-value : {result.metadata.get('min_p_value')}")
        print(f"         n_periods   : {result.metadata.get('n_periods')}")